# Trends in Fitbit steps in countries with prevalent Fitbit users

Author: Bram Schonfeldt

This notebook compares trends in Fitbit steps to trends in self-reported
physical activity prevalence obtained from prior work by Strain et al., 2024,
"National, Regional, and Global Trends in Insufficient Physical Activity among
Adults from 2000 to 2022: A Pooled Analysis of 507 Population-Based Surveys with
5.7 Million Participants." The Lancet. Global Health.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import pickle as pkl
import sys
from time import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.axes
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit
import seaborn as sns
import statsmodels.api as sm

# Set rcParams
plt.rcdefaults()
plt.rcParams['figure.dpi'] = 200

sys.path.insert(0, os.path.abspath('../..'))
import stratify as strat

We previously ran code which extracts all steps within a certain time period
and aggregates the number of readings, the sum, and sum of squared steps per
age-group, sex, year, and country. To reload the saved statistics from the
precomputed checkpoint (`override_hash='a36731bea99396ebe673f464d73ada56'`), we
supply the matching cohort parameters along with the path to the saved data and
the UN population estimates (distinct for males and females). Finally, we
specify the path to the combined step estimates from the NHNS-J.

In [ ]:
# Start and end dates for the analysis cohort
start_date = '2015-01-01'
end_date = '2026-01-01'

# Number of shards used for parallel processing
n_shards = 100

# Base paths for internal and public datasets
internal_dir = strat.get_internal_data_dir()
public_dir = strat.get_public_data_dir()

# UN Population 5-year age group data paths
un_pop_male_path = (
    f'{public_dir}/WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx'
)
un_pop_female_path = (
    f'{public_dir}/WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx'
)

# Path to pre-extracted Japanese step counts
japan_step_path = f'{public_dir}/2015_2024_japan_steps.pkl'

Load in the data...

In [ ]:
# Ensure configuration variables are defined even if Cell 3 was skipped
if 'internal_dir' not in globals():
  start_date = '2015-01-01'
  end_date = '2026-01-01'
  n_shards = 100
  internal_dir = strat.get_internal_data_dir()
  public_dir = strat.get_public_data_dir()
  un_pop_male_path = (
      f'{public_dir}/WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx'
  )
  un_pop_female_path = (
      f'{public_dir}/WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx'
  )


# Shared helper function for 10-year age midpoints (capped at 75 for 70+)
def midage_10(age: int) -> int:
  return age // 10 * 10 + 5 if age < 70 else 75


# 1a. Load global pre-computed stratified step grid from cached shards.
step_grid = strat.StratifySteps.from_shards(
    path_base=internal_dir,
    start_date=start_date,
    end_date=end_date,
    n_shards=n_shards,
    un_pop_male_path=un_pop_male_path,
    un_pop_female_path=un_pop_female_path,
    min_days_per_user=1,
    min_age=20,
    max_age=104,
    min_steps=None,
    max_steps=172800,
    target_country=None,
    start_month=None,
    stop_month=None,
    exclude_days_of_week=None,
    override_hash='a36731bea99396ebe673f464d73ada56',
)

# 1b. Load WHO physical activity trends dataset (Strain et al., 2024)
who_trends_path = f'{public_dir}/who_trends.csv'
try:
  with strat.open_file(who_trends_path, 'r') as f:
    who_df = pd.read_csv(f)
except Exception:
  who_df = pd.read_csv('who_trends.csv')


def parse_prevalence(val: Any) -> float:
  if isinstance(val, str):
    return float(val.split('%')[0])
  return np.nan


for sex_raw, sex_val in [('Men', 'male'), ('Women', 'female')]:
  for yr in [2010, 2022]:
    col = f'{sex_raw}: {yr} Prevalence (95% UI)'
    who_df[f'{sex_val}_insuff_{yr}'] = who_df[col].apply(parse_prevalence)
    who_df[f'{sex_val}_suff_{yr}'] = 100.0 - who_df[f'{sex_val}_insuff_{yr}']

for sex_val in ['male', 'female']:
  who_df[f'{sex_val}_pa_rel_change'] = (
      (who_df[f'{sex_val}_suff_2022'] - who_df[f'{sex_val}_suff_2010'])
      / who_df[f'{sex_val}_suff_2010']
      * 100
  )

who_trends_processed = who_df[[
    'Country / Territory',
    'male_pa_rel_change',
    'female_pa_rel_change',
]].copy()
who_trends_processed.rename(
    columns={'Country / Territory': 'country'}, inplace=True
)

who_countries = set(who_df['Country / Territory'].unique())

fb_df = step_grid.results_grid.reset_index()
fb_df['midage_10'] = fb_df['midage'].apply(midage_10)
fb_df['sex'] = fb_df['gender'].replace({0: 'male', 1: 'female'})

un_df = step_grid.un_pop.copy()
un_df['midage_10'] = un_df['age'].apply(midage_10)
un_df['sex'] = un_df['gender'].replace({0: 'male', 1: 'female'})

fb_subset = fb_df[(fb_df['year'] >= 2015) & (fb_df['year'] <= 2022)].copy()

# Criterion 1: >= 30 readings in ALL 12 demographic bins per country-year
fb_group_counts = (
    fb_subset.groupby(['country', 'year', 'sex', 'midage_10'])['readings']
    .sum()
    .reset_index()
)
low_count_mask = fb_group_counts['readings'] < 30
fb_group_counts['country_year'] = (
    fb_group_counts['country'].astype(str)
    + '_'
    + fb_group_counts['year'].astype(str)
)
low_count_keys = fb_group_counts[low_count_mask]['country_year'].unique()

fb_subset['country_year'] = (
    fb_subset['country'].astype(str) + '_' + fb_subset['year'].astype(str)
)
fb_filtered = fb_subset[~fb_subset['country_year'].isin(low_count_keys)].copy()

# Criterion 2: >= 5 valid years between 2015 and 2022
valid_years = (
    fb_filtered.groupby(['country', 'sex'])['year'].nunique().reset_index()
)

target_countries = []
for country in fb_filtered['country'].unique():
  m_yrs = valid_years[
      (valid_years['country'] == country) & (valid_years['sex'] == 'male')
  ]['year'].values
  f_yrs = valid_years[
      (valid_years['country'] == country) & (valid_years['sex'] == 'female')
  ]['year'].values
  if len(m_yrs) > 0 and len(f_yrs) > 0 and m_yrs[0] >= 5 and f_yrs[0] >= 5:
    if country in who_countries:
      target_countries.append(country)

target_countries = sorted(target_countries)

mask_world = un_df['country'].str.contains('world', case=False, na=False)
un_world = un_df[mask_world].copy()
un_world = un_world[(un_world['year'] >= 2015) & (un_world['year'] <= 2022)]

world_average = (
    un_world.groupby(['sex', 'midage_10'])['Count']
    .mean()
    .reset_index()
    .rename(columns={'Count': 'world_weight'})
)

fb_target_subset = fb_df[
    (fb_df['country'].isin(target_countries))
    & (fb_df['year'] >= 2015)
    & (fb_df['year'] <= 2022)
].copy()

stratum_means = (
    fb_target_subset.groupby(['country', 'year', 'sex', 'midage_10'])
    .agg(total_steps=('steps', 'sum'), total_readings=('readings', 'sum'))
    .reset_index()
)
stratum_means['mean_steps'] = (
    stratum_means['total_steps'] / stratum_means['total_readings']
)

stratum_weighted = stratum_means.merge(
    world_average, on=['sex', 'midage_10'], how='inner'
)


def compute_age_standardized_mean(df: pd.DataFrame) -> float:
  return (df['mean_steps'] * df['world_weight']).sum() / df[
      'world_weight'
  ].sum()


std_yearly = (
    stratum_weighted.groupby(['country', 'year', 'sex'])
    .apply(compute_age_standardized_mean, include_groups=False)
    .reset_index(name='std_mean_steps')
)

fitbit_model_results = []
for country in std_yearly['country'].unique():
  for g in ['male', 'female']:
    c_df = std_yearly[
        (std_yearly['country'] == country) & (std_yearly['sex'] == g)
    ].sort_values('year')
    if len(c_df) < 5:
      continue

    years = c_df['year'].values
    steps = c_df['std_mean_steps'].values

    slope, intercept, r_value, p_value, std_err = stats.linregress(years, steps)
    pred_2010 = intercept + slope * 2010
    pred_2022 = intercept + slope * 2022
    modelled_rel_change = (
        (pred_2010 - pred_2010 + pred_2022 - pred_2010) / pred_2010 * 100
    )

    fitbit_model_results.append({
        'country': country,
        'sex': g,
        'fitbit_modelled_rel_change': modelled_rel_change,
    })

fitbit_model_df = pd.DataFrame(fitbit_model_results)
fb_pivot_sex = fitbit_model_df.pivot(
    index='country', columns='sex', values='fitbit_modelled_rel_change'
).reset_index()
fb_pivot_sex.rename(
    columns={
        'male': 'male_fitbit_rel_change',
        'female': 'female_fitbit_rel_change',
    },
    inplace=True,
)

comparison_clean = (
    pd.merge(fb_pivot_sex, who_trends_processed, on='country', how='inner')
    .dropna()
    .copy()
)

fb_df_2025 = fb_df[
    (fb_df['country'].isin(target_countries)) & (fb_df['year'] <= 2025)
].copy()
fb_grouped_traj = (
    fb_df_2025.groupby(['country', 'year', 'sex', 'midage_10'])[
        ['steps', 'steps_sq', 'readings']
    ]
    .sum()
    .reset_index()
)
fb_grouped_traj['mean_steps'] = (
    fb_grouped_traj['steps'] / fb_grouped_traj['readings']
)
fb_grouped_traj['var_steps'] = (
    fb_grouped_traj['steps_sq'] / fb_grouped_traj['readings']
) - fb_grouped_traj['mean_steps'] ** 2
fb_grouped_traj['var_steps'] = np.maximum(fb_grouped_traj['var_steps'], 0)
fb_grouped_traj['std_steps'] = np.sqrt(fb_grouped_traj['var_steps'])

un_world_2025 = un_world.copy()
world_avg_2025 = (
    un_world_2025.groupby(['sex', 'midage_10'])['Count']
    .mean()
    .reset_index()
    .rename(columns={'Count': 'avg_world_count'})
)

merged_traj = fb_grouped_traj.merge(
    world_avg_2025, on=['sex', 'midage_10'], how='inner'
)

B_RUNS = 1000
rng = np.random.default_rng(42)

trend_results = {
    'male': {'mean': {}, 'ci_lower': {}, 'ci_upper': {}},
    'female': {'mean': {}, 'ci_lower': {}, 'ci_upper': {}},
}

for g in ['male', 'female']:
  for country in target_countries:
    c_data = (
        merged_traj[
            (merged_traj['country'] == country) & (merged_traj['sex'] == g)
        ]
        .copy()
        .reset_index(drop=True)
    )
    if c_data.empty:
      continue

    means = c_data['mean_steps'].values[:, np.newaxis]
    stds = c_data['std_steps'].values[:, np.newaxis]
    ns = c_data['readings'].values[:, np.newaxis]
    se = stds / np.sqrt(ns)
    df_degrees = np.maximum(ns - 1, 1)

    t_draws = rng.standard_t(df=df_degrees, size=(len(c_data), B_RUNS))
    boot_samples = np.clip(means + t_draws * se, a_min=0, a_max=None)

    y_list = sorted(c_data['year'].unique())
    aggregated = np.full((len(y_list), B_RUNS), np.nan)

    for idx, year in enumerate(y_list):
      y_mask = c_data['year'].values == year
      if np.any(y_mask):
        y_samples = boot_samples[y_mask]
        raw_w = c_data['avg_world_count'].values[y_mask]
        w_sum = raw_w.sum()
        if w_sum > 0:
          aggregated[idx] = np.sum(
              y_samples * (raw_w / w_sum)[:, np.newaxis], axis=0
          )

    valid_mask = ~np.all(np.isnan(aggregated), axis=1)
    if np.any(valid_mask):
      years_v = np.array(y_list)[valid_mask]
      samples_v = aggregated[valid_mask]

      trend_results[g]['mean'][country] = pd.Series(
          np.mean(samples_v, axis=1), index=years_v
      )
      trend_results[g]['ci_lower'][country] = pd.Series(
          np.percentile(samples_v, 2.5, axis=1), index=years_v
      )
      trend_results[g]['ci_upper'][country] = pd.Series(
          np.percentile(samples_v, 97.5, axis=1), index=years_v
      )

print(
    'Successfully loaded and cached data for 28-country WHO correlation'
    ' (Figure 2) and trajectories (Figure 3).'
)
print(f'28 Target Countries ({len(target_countries)}): {target_countries}')

In [ ]:
# Mapping of country names to official 2-letter ISO codes
COUNTRY_ISO_MAP = {
    'Australia': 'AU',
    'Austria': 'AT',
    'Belgium': 'BE',
    'Canada': 'CA',
    'Chile': 'CL',
    'Czech Republic': 'CZ',
    'Czechia': 'CZ',
    'Denmark': 'DK',
    'Finland': 'FI',
    'France': 'FR',
    'Germany': 'DE',
    'Greece': 'GR',
    'Hungary': 'HU',
    'India': 'IN',
    'Ireland': 'IE',
    'Israel': 'IL',
    'Italy': 'IT',
    'Japan': 'JP',
    'Mexico': 'MX',
    'Netherlands': 'NL',
    'New Zealand': 'NZ',
    'Norway': 'NO',
    'Philippines': 'PH',
    'Poland': 'PL',
    'Portugal': 'PT',
    'Republic of Korea': 'KR',
    'South Korea': 'KR',
    'South Africa': 'ZA',
    'Spain': 'ES',
    'Sweden': 'SE',
    'Switzerland': 'CH',
    'United Kingdom': 'UK',
    'United States of America': 'US',
}


def get_iso_code(c_name: str) -> str:
  """Returns the 2-letter ISO code for a given country name."""
  return COUNTRY_ISO_MAP.get(c_name, c_name[:2].upper())


country_sample_records = []
for c in target_countries:
  iso = get_iso_code(c)
  sub_c = fb_df[fb_df['country'] == c]
  n_2015 = sub_c[sub_c['year'] == 2015]['readings'].sum()
  n_2022 = sub_c[sub_c['year'] == 2022]['readings'].sum()
  country_sample_records.append({
      'ISO Code': iso,
      'Country / Territory': c,
      'Fitbit Users (2015)': int(n_2015),
      'Fitbit Users (2022)': int(n_2022),
  })

df_country_samples = pd.DataFrame(country_sample_records).sort_values(
    'Country / Territory'
)

print('=' * 95)
print(
    'FITBIT SAMPLE SIZES BY COUNTRY (2015 vs 2022) ACROSS 28 INCLUDED COUNTRIES'
)
print('=' * 95)
display(
    df_country_samples.style.format({
        'Fitbit Users (2015)': '{:,}',
        'Fitbit Users (2022)': '{:,}',
    })
)

# LaTeX Code for Country Sample Sizes Table
latex_country_lines = [
    r'\begin{table}[htbp]',
    r'\centering',
    (
        r'\caption{\textbf{Included countries, ISO-2 abbreviations, and annual'
        r' Fitbit sample sizes in 2015 and 2022.} Total number of eligible'
        r' Fitbit participants meeting quality and wear-time inclusion criteria'
        r' ($\ge 1$ valid wear day in the study window) in 2015 and 2022 across'
        r' the 28 countries analyzed.}'
    ),
    r'\label{tab:country_fitbit_samples}',
    r'\begin{tabular}{llrr}',
    r'\toprule',
    (
        r'\textbf{ISO} & \textbf{Country / Territory} & \textbf{Fitbit Users'
        r' (2015)} & \textbf{Fitbit Users (2022)} \\'
    ),
    r'\midrule',
]
for _, r in df_country_samples.iterrows():
  iso_v = r['ISO Code']
  c_v = r['Country / Territory']
  n15_v = f"{r['Fitbit Users (2015)']:,}"
  n22_v = f"{r['Fitbit Users (2022)']:,}"
  latex_country_lines.append(f'{iso_v} & {c_v} & {n15_v} & {n22_v} \\\\')
latex_country_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
])

print('\n' + '=' * 95)
print('LATEX CODE FOR COUNTRY SAMPLE SIZE TABLE:')
print('=' * 95)
print('\n'.join(latex_country_lines))
print('=' * 95)

## Global & 28-Country Analysis

In this section, we present the main figures analyzing demographic
representation, relative physical activity trends, and longitudinal step
trajectories across the 28 countries meeting our inclusion criteria
(at least 30 users per demographic stratum, 5 valid years between 2015–2022,
and matching survey data).

In [ ]:
# Format: 'Country Name' or 'ISO Code': (nudge_x_percent, nudge_y_percent)
MANUAL_LABEL_OFFSETS_MALE = {
    'US': (-1, -3),
    'SE': (0.5, 1.5),
    'DK': (1.5, 1),
    'UK': (0, 3),
    'FR': (-1.5, 2),
    'AT': (-1.5, 2),
    'CH': (2, -2),
    'IL': (0, -3),
    'IT': (-2, 2),
    'PT': (2, 0),
    'SI': (-2, -2),
}

MANUAL_LABEL_OFFSETS_FEMALE = {
    'IL': (0, 2),
    'ZA': (0, -2),
    'CA': (2, 0),
    'CH': (1, -2.5),
    'DE': (0, -2),
    'NL': (-2, 2),
    'SE': (-2, 0),
    'FR': (1, 0),
}

comparison_plot_df = comparison_clean.copy()
comparison_plot_df['iso_code'] = comparison_plot_df['country'].apply(
    get_iso_code
)

# Compute symmetric axis limits centered at 0 across sexes
x_all = pd.concat([
    comparison_plot_df['male_fitbit_rel_change'],
    comparison_plot_df['female_fitbit_rel_change'],
])
y_all = pd.concat([
    comparison_plot_df['male_pa_rel_change'],
    comparison_plot_df['female_pa_rel_change'],
])

max_abs_x = max(abs(x_all.min()), abs(x_all.max()))
max_abs_y = max(abs(y_all.min()), abs(y_all.max()))

x_bound = np.ceil((max_abs_x + 1.0) / 10) * 10
y_bound = np.ceil((max_abs_y + 1.0) / 10) * 10

shared_xlim = (-x_bound, x_bound)
shared_ylim = (-y_bound, y_bound)

fig, axs = plt.subplots(
    1, 2, figsize=(7.0, 3.4), sharey=True, sharex=True, layout='constrained'
)

for i, sex_val in enumerate(['male', 'female']):
  ax = axs[i]
  x_val = comparison_plot_df[f'{sex_val}_fitbit_rel_change']
  y_val = comparison_plot_df[f'{sex_val}_pa_rel_change']

  offsets_dict = (
      MANUAL_LABEL_OFFSETS_MALE
      if sex_val == 'male'
      else MANUAL_LABEL_OFFSETS_FEMALE
  )

  # Fit linear regression trendline
  if len(x_val) > 2:
    m, b, r, p, _ = stats.linregress(x_val, y_val)
    r2 = r**2
    x_seq = np.linspace(shared_xlim[0], shared_xlim[1], 100)
    p_str = '$p < 0.001$' if p < 0.001 else f'$p = {p:.3f}$'
    ax.plot(
        x_seq,
        m * x_seq + b,
        color='crimson',
        linestyle='--',
        linewidth=1.2,
        label=f'$R^2 = {r2:.2f}$ ({p_str})',
    )
    ax.legend(
        loc='lower right',
        bbox_to_anchor=(0.99, 0.06),
        frameon=False,
        fontsize=8,
    )

  # Plot country scatter points
  ax.scatter(
      x_val,
      y_val,
      color='#333333',
      edgecolor='none',
      alpha=0.5,
      s=18,
  )

  # Annotate country labels
  for _, row in comparison_plot_df.iterrows():
    c_name = row['country']
    c_code = row['iso_code']
    cx = row[f'{sex_val}_fitbit_rel_change']
    cy = row[f'{sex_val}_pa_rel_change']

    if c_name in offsets_dict or c_code in offsets_dict:
      nudge_x, nudge_y = offsets_dict.get(c_name, offsets_dict.get(c_code))
      label_x = cx + nudge_x
      label_y = cy + nudge_y

      ax.annotate(
          c_code,
          xy=(cx, cy),
          xytext=(label_x, label_y),
          textcoords='data',
          fontsize=6.5,
          color='#333333',
          ha='center' if nudge_x == 0 else ('left' if nudge_x > 0 else 'right'),
          va='center' if nudge_y == 0 else ('bottom' if nudge_y > 0 else 'top'),
          arrowprops=dict(
              arrowstyle='-',
              color='#666666',
              linewidth=0.6,
              shrinkA=0,
              shrinkB=0,
              alpha=0.75,
          ),
      )
    else:
      ax.annotate(
          c_code,
          (cx, cy),
          textcoords='offset points',
          xytext=(0, 1),
          fontsize=6.5,
          color='#333333',
          ha='center',
          va='bottom',
      )

  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.axhline(0, color='gray', linestyle=':', alpha=0.4, linewidth=0.8)
  ax.axvline(0, color='gray', linestyle=':', alpha=0.4, linewidth=0.8)
  ax.set_xlim(shared_xlim)
  ax.set_ylim(shared_ylim)
  ax.set_title(f'{sex_val.capitalize()}')

  # Granular symmetric ticks every 10% on both x and y axes
  ax.xaxis.set_major_locator(MultipleLocator(10))
  ax.yaxis.set_major_locator(MultipleLocator(10))

  ax.text(
      0.02,
      1.0,
      '↑ Increasing PA',
      transform=ax.transAxes,
      ha='left',
      va='top',
      color='dimgray',
      fontstyle='italic',
  )
  ax.text(
      1.0,
      0.005,
      'Increasing steps →',
      transform=ax.transAxes,
      ha='right',
      va='bottom',
      color='dimgray',
      fontstyle='italic',
  )

fig.suptitle(
    'Twelve year changes in Fitbit steps and sufficient physical activity'
    ' prevalence'
)
fig.supxlabel(
    'Change in Fitbit steps (%)\n(2022 estimate − 2010 estimate) / 2010'
    ' estimate',
    fontsize=10,
)
fig.supylabel(
    'Change in\nsufficient PA\nprevalence (%)',
    rotation=0,
    fontsize=10,
    ha='right',
    x=-0.01,
)
plt.show()

In [ ]:
# Countries selected for visual highlighting and annotation
HIGHLIGHT_COUNTRIES = {
    'Japan',
    'United Kingdom',
    'India',
    'Ireland',
    'Belgium',
    'Philippines',
}

comparison_plot_df['is_highlight'] = comparison_plot_df['country'].isin(
    HIGHLIGHT_COUNTRIES
)

# Compute symmetric axis limits centered at 0 across sexes
x_all = pd.concat([
    comparison_plot_df['male_fitbit_rel_change'],
    comparison_plot_df['female_fitbit_rel_change'],
])
y_all = pd.concat([
    comparison_plot_df['male_pa_rel_change'],
    comparison_plot_df['female_pa_rel_change'],
])

max_abs_x = max(abs(x_all.min()), abs(x_all.max()))
max_abs_y = max(abs(y_all.min()), abs(y_all.max()))

x_bound = np.ceil((max_abs_x + 1.0) / 10) * 10
y_bound = np.ceil((max_abs_y + 1.0) / 10) * 10

shared_xlim = (-x_bound, x_bound)
shared_ylim = (-y_bound, y_bound)

fig, axs = plt.subplots(
    1, 2, figsize=(7.0, 3.4), sharey=True, sharex=True, layout='constrained'
)

for i, sex_val in enumerate(['male', 'female']):
  ax = axs[i]
  x_val = comparison_plot_df[f'{sex_val}_fitbit_rel_change']
  y_val = comparison_plot_df[f'{sex_val}_pa_rel_change']

  highlight_mask = comparison_plot_df['is_highlight']
  bg_mask = ~highlight_mask

  # Fit linear regression trendline
  if len(x_val) > 2:
    m, b, r, p, _ = stats.linregress(x_val, y_val)
    r2 = r**2
    x_seq = np.linspace(shared_xlim[0], shared_xlim[1], 100)
    p_str = '$p < 0.001$' if p < 0.001 else f'$p = {p:.3f}$'
    ax.plot(
        x_seq,
        m * x_seq + b,
        color='crimson',
        linestyle='--',
        linewidth=1.2,
        label=f'$R^2={r2:.2f}$ ({p_str})',
    )
    ax.legend(
        loc='lower right',
        bbox_to_anchor=(0.99, 0.06),
        frameon=False,
        fontsize=8,
    )

  # Plot unannotated background countries (light gray)
  ax.scatter(
      x_val[bg_mask],
      y_val[bg_mask],
      color='#bbbbbb',
      edgecolor='none',
      s=16,
  )

  # Plot highlighted target countries (dark charcoal, larger marker size)
  ax.scatter(
      x_val[highlight_mask],
      y_val[highlight_mask],
      color='#111111',
      edgecolor='none',
      s=16,
  )

  # Annotate highlighted countries with full country names centered directly above scatter points
  highlight_df = comparison_plot_df[highlight_mask]
  for _, row in highlight_df.iterrows():
    c_full_name = row['country']
    cx = row[f'{sex_val}_fitbit_rel_change']
    cy = row[f'{sex_val}_pa_rel_change']

    ax.annotate(
        c_full_name,
        (cx, cy),
        textcoords='offset points',
        xytext=(0, 2),
        fontsize=7.5,
        color='#111111',
        ha='center',
        va='bottom',
    )

  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.axhline(0, color='gray', linestyle=':', alpha=0.4, linewidth=0.8)
  ax.axvline(0, color='gray', linestyle=':', alpha=0.4, linewidth=0.8)
  ax.set_xlim(shared_xlim)
  ax.set_ylim(shared_ylim)
  ax.set_title(f'{sex_val.capitalize()}')

  # Granular symmetric ticks every 10% on both x and y axes
  ax.xaxis.set_major_locator(MultipleLocator(10))
  ax.yaxis.set_major_locator(MultipleLocator(10))

  ax.text(
      0.02,
      1.0,
      '↑ Increasing PA',
      transform=ax.transAxes,
      ha='left',
      va='top',
      color='dimgray',
      fontstyle='italic',
      fontsize=8,
  )
  ax.text(
      1.0,
      0.005,
      'Increasing steps →',
      transform=ax.transAxes,
      ha='right',
      va='bottom',
      color='dimgray',
      fontstyle='italic',
      fontsize=8,
  )

fig.suptitle(
    'Twelve year changes in Fitbit steps and sufficient physical activity'
    ' prevalence'
)
fig.supxlabel(
    'Change in Fitbit steps (%)\n(2022 estimate − 2010 estimate) / 2010'
    ' estimate',
    fontsize=10,
)
fig.supylabel(
    'Change in\nsufficient PA\nprevalence (%)',
    rotation=0,
    fontsize=10,
    ha='right',
    x=-0.01,
)
plt.show()

In [ ]:
# Model Equations & Regression Statistics
print('=' * 85)
print(
    'REGRESSION MODEL STATISTICS: FITBIT STEP % CHANGE vs. WHO PA PREVALENCE %'
    ' CHANGE'
)
print('=' * 85)

for sex_val in ['male', 'female']:
  x_val = comparison_plot_df[f'{sex_val}_fitbit_rel_change']
  y_val = comparison_plot_df[f'{sex_val}_pa_rel_change']
  slope, intercept, r_value, p_value, std_err = stats.linregress(x_val, y_val)
  r_squared = r_value**2
  print(f'{sex_val.upper()}:')
  print(
      f'  Model Equation:    ΔPA_suff = {slope:.4f} * ΔFitbit_Steps +'
      f' {intercept:.4f}'
  )
  print(f'  R-squared (R²):    {r_squared:.4f} (r = {r_value:.4f})')
  print(f'  p-value:           {p_value:.6e}')
  print(f'  Standard Error:    {std_err:.4f}')
  print(f'  Sample Size (N):   {len(x_val)} countries\n')
print('=' * 85)

In [ ]:
# Country-by-Country Comparison of Physical Activity Prevalence & Step Count Trends
comparison_records = []

for c in sorted(target_countries):
  iso = get_iso_code(c)
  who_row = who_df[who_df['Country / Territory'] == c]
  if who_row.empty:
    continue

  for s_fb, s_label in [('female', 'Female'), ('male', 'Male')]:
    s_who = s_fb

    # WHO Surveillance Values (12-year window 2010 -> 2022)
    who_2010 = who_row[f'{s_who}_suff_2010'].values[0]
    who_2022 = who_row[f'{s_who}_suff_2022'].values[0]
    who_change = who_row[f'{s_who}_pa_rel_change'].values[0]
    who_ann_change = who_change / 12.0

    # Fitbit Steps: OLS Linear Trend across all available annual data points (2015-2022)
    c_df = std_yearly[
        (std_yearly['country'] == c) & (std_yearly['sex'] == s_fb)
    ].sort_values('year')
    if len(c_df) < 5:
      continue

    years = c_df['year'].values
    steps = c_df['std_mean_steps'].values
    slope, intercept, r_val, p_val, _ = stats.linregress(years, steps)

    pred_2010 = intercept + slope * 2010
    pred_2015 = intercept + slope * 2015
    pred_2022 = intercept + slope * 2022

    # 12-Year Modeled Relative Change informed by OLS trend slope (beta * 12 / pred_2010)
    fb_change = (pred_2022 - pred_2010) / pred_2010 * 100
    fb_ann_rate = slope / pred_2010 * 100

    # Empirical observed steps in 2015 and 2022
    obs_2015 = (
        c_df[c_df['year'] == 2015]['std_mean_steps'].values[0]
        if 2015 in c_df['year'].values
        else np.nan
    )
    obs_2022 = (
        c_df[c_df['year'] == 2022]['std_mean_steps'].values[0]
        if 2022 in c_df['year'].values
        else np.nan
    )

    discrepancy = fb_change - who_change
    abs_discrepancy = abs(discrepancy)

    comparison_records.append({
        'ISO': iso,
        'Country': c,
        'Sex': s_label,
        'WHO Suff PA 2010 (%)': who_2010,
        'WHO Suff PA 2022 (%)': who_2022,
        'Δ WHO PA (%)': who_change,
        'WHO Ann Rate (%/yr)': who_ann_change,
        'Fitbit Slope β (steps/yr)': slope,
        'Fitbit Steps 2010 (Model)': pred_2010,
        'Fitbit Steps 2015 (Obs)': obs_2015,
        'Fitbit Steps 2022 (Obs)': obs_2022,
        'Fitbit Steps 2022 (Model)': pred_2022,
        'Δ Fitbit Steps (%)': fb_change,
        'Fitbit Ann Rate (%/yr)': fb_ann_rate,
        'Discrepancy (ΔSteps − ΔPA, pp)': discrepancy,
        '|Discrepancy| (pp)': abs_discrepancy,
    })

df_all_comparison = pd.DataFrame(comparison_records)

for s_label in ['Female', 'Male']:
  df_s = (
      df_all_comparison[df_all_comparison['Sex'] == s_label]
      .sort_values('Δ Fitbit Steps (%)', ascending=False)
      .reset_index(drop=True)
  )

  print('=' * 135)
  print(
      f'{s_label.upper()}: COMPARISON OF WHO PHYSICAL ACTIVITY & FITBIT STEP'
      ' CHANGES (2010-2022)'
  )
  print(
      'Note: Fitbit 12-year changes are modeled via OLS regression over all'
      ' available years (2015-2022): ΔSteps = (12 * β) / Pred_2010.'
  )
  print('=' * 135)

  display(
      df_s[[
          'ISO',
          'Country',
          'WHO Suff PA 2010 (%)',
          'WHO Suff PA 2022 (%)',
          'Δ WHO PA (%)',
          'Fitbit Slope β (steps/yr)',
          'Fitbit Steps 2010 (Model)',
          'Fitbit Steps 2015 (Obs)',
          'Fitbit Steps 2022 (Obs)',
          'Fitbit Steps 2022 (Model)',
          'Δ Fitbit Steps (%)',
          'Discrepancy (ΔSteps − ΔPA, pp)',
          '|Discrepancy| (pp)',
      ]].style.format({
          'WHO Suff PA 2010 (%)': '{:.1f}%',
          'WHO Suff PA 2022 (%)': '{:.1f}%',
          'Δ WHO PA (%)': '{:+.1f}%',
          'Fitbit Slope β (steps/yr)': '{:+.1f}',
          'Fitbit Steps 2010 (Model)': '{:,.0f}',
          'Fitbit Steps 2015 (Obs)': '{:,.0f}',
          'Fitbit Steps 2022 (Obs)': '{:,.0f}',
          'Fitbit Steps 2022 (Model)': '{:,.0f}',
          'Δ Fitbit Steps (%)': '{:+.1f}%',
          'Discrepancy (ΔSteps − ΔPA, pp)': '{:+.1f}',
          '|Discrepancy| (pp)': '{:.1f}',
      })
  )

  # Highlight top 3 most similar and top 3 most dissimilar countries
  df_by_sim = df_s.sort_values('|Discrepancy| (pp)')
  most_similar = df_by_sim.head(3)
  most_dissimilar = df_by_sim.tail(3)

  print(
      f'\n--- {s_label}: Most Similar Trends (Smallest Discrepancy between'
      ' ΔSteps and ΔPA) ---'
  )
  for _, r in most_similar.iterrows():
    print(
        f"  • {r['Country']} ({r['ISO']}): ΔFitbit Steps ="
        f" {r['Δ Fitbit Steps (%)']:+.1f}% (β ="
        f" {r['Fitbit Slope β (steps/yr)']:+.1f} steps/yr), ΔWHO PA ="
        f" {r['Δ WHO PA (%)']:+.1f}% (Diff ="
        f" {r['Discrepancy (ΔSteps − ΔPA, pp)']:+.1f} pp)"
    )

  print(
      f'\n--- {s_label}: Most Dissimilar Trends (Largest Discrepancy between'
      ' ΔSteps and ΔPA) ---'
  )
  for _, r in most_dissimilar.iterrows():
    print(
        f"  • {r['Country']} ({r['ISO']}): ΔFitbit Steps ="
        f" {r['Δ Fitbit Steps (%)']:+.1f}% (β ="
        f" {r['Fitbit Slope β (steps/yr)']:+.1f} steps/yr), ΔWHO PA ="
        f" {r['Δ WHO PA (%)']:+.1f}% (Diff ="
        f" {r['Discrepancy (ΔSteps − ΔPA, pp)']:+.1f} pp)"
    )
  print('\n')

for s_label in ['Female', 'Male']:
  df_s = (
      df_all_comparison[df_all_comparison['Sex'] == s_label]
      .sort_values('Δ Fitbit Steps (%)', ascending=False)
      .reset_index(drop=True)
  )

  latex_lines = [
      r'\begin{table*}[htbp]',
      r'\centering',
      r'\setlength{\tabcolsep}{4.5pt}',
      (
          f'\\caption{{\\textbf{{{s_label}: Country-level changes in WHO'
          ' physical activity prevalence and Fitbit longitudinal step trends'
          ' (2010--2022).}} National prevalence of sufficient physical'
          ' activity from WHO surveillance (Strain et al., 2024) compared'
          ' against longitudinal age-standardized Fitbit step trends across the'
          ' 28 included countries, ordered by relative change in Fitbit steps.'
          r' $\beta$ denotes the annual regression slope (steps/year) fitted'
          ' across all available annual cohorts between 2015 and 2022.}'
      ),
      f'\\label{{tab:who_fitbit_trend_comparison_{s_label.lower()}}}',
      r'\begin{tabular}{ll ccc cccc c}',
      r'\toprule',
      (
          r' & & \multicolumn{3}{c}{\textbf{WHO Sufficient PA Prevalence}} &'
          r' \multicolumn{4}{c}{\textbf{Fitbit Daily Steps (Longitudinal'
          r' Trend)}} & \textbf{Discrepancy} \\'
      ),
      r'\cmidrule(lr){3-5} \cmidrule(lr){6-9} \cmidrule(lr){10-10}',
      (
          r'\textbf{ISO} & \textbf{Country} & \textbf{2010 (\%)} &'
          r' \textbf{2022 (\%)} & \textbf{$\Delta$ PA (\%)} & \textbf{$\beta$'
          r' (steps/yr)} & \textbf{2010 (Model)} & \textbf{2022 (Model)} &'
          r' \textbf{$\Delta$ Steps (\%)} & \textbf{$\Delta$Steps $-$'
          r' $\Delta$PA} \\'
      ),
      r'\midrule',
  ]

  for _, r in df_s.iterrows():
    iso_v = r['ISO']
    c_v = r['Country']
    who10 = f"{r['WHO Suff PA 2010 (%)']:.1f}\\%"
    who22 = f"{r['WHO Suff PA 2022 (%)']:.1f}\\%"
    who_ch = f"{r['Δ WHO PA (%)']:+.1f}\\%"

    beta_v = f"{r['Fitbit Slope β (steps/yr)']:+.1f}"
    fb10 = f"{r['Fitbit Steps 2010 (Model)']:,.0f}"
    fb22 = f"{r['Fitbit Steps 2022 (Model)']:,.0f}"
    fb_ch = f"{r['Δ Fitbit Steps (%)']:+.1f}\\%"
    disc = f"{r['Discrepancy (ΔSteps − ΔPA, pp)']:+.1f}"

    latex_lines.append(
        f'{iso_v} & {c_v} & {who10} & {who22} & {who_ch} & {beta_v} & {fb10}'
        f' & {fb22} & {fb_ch} & {disc} \\\\'
    )

  latex_lines.extend([
      r'\bottomrule',
      r'\end{tabular}',
      r'\end{table*}',
  ])

  print('=' * 135)
  print(f'LATEX TABLE CODE ({s_label.upper()}):')
  print('=' * 135)
  print('\n'.join(latex_lines))
  print('=' * 135 + '\n')

## Average steps counts from 2015 to 2025 across countries with demographic coverage

We present longitudinal step count trajectories from 2015 to 2025 across the 28
target countries broken down by different factors:
1. Sex (Male, Female) and Age Group (20–39, 40–59, 60+).
2. Geographic Region (Europe, Americas, Asia-Pacific, Middle East, Africa).
3. World Bank Income Group (High Income vs. Middle Income).

In [ ]:
fig = plt.figure(figsize=(7.0, 5.2))
gs = gridspec.GridSpec(2, 12, figure=fig, hspace=0.55, wspace=0.45, top=0.88)

ax_sex_m = fig.add_subplot(gs[0, 0:6])
ax_sex_w = fig.add_subplot(gs[0, 6:12], sharey=ax_sex_m)

ax_age_1 = fig.add_subplot(gs[1, 0:4])
ax_age_2 = fig.add_subplot(gs[1, 4:8], sharey=ax_age_1)
ax_age_3 = fig.add_subplot(gs[1, 8:12], sharey=ax_age_1)

all_demo_axes = [ax_sex_m, ax_sex_w, ax_age_1, ax_age_2, ax_age_3]
leftmost_demo = [ax_sex_m, ax_age_1]

for sex_val, ax, title_str in [
    ('male', ax_sex_m, 'A1. Male'),
    ('female', ax_sex_w, 'A2. Female'),
]:
  sub_df = fb_df[
      (fb_df['country'].isin(target_countries))
      & (fb_df['year'] <= 2025)
      & (fb_df['sex'] == sex_val)
  ].copy()
  c_yearly = (
      sub_df.groupby(['country', 'year'])
      .agg(total_steps=('steps', 'sum'), total_readings=('readings', 'sum'))
      .reset_index()
  )
  c_yearly['mean_steps'] = c_yearly['total_steps'] / c_yearly['total_readings']

  for country, c_data in c_yearly.groupby('country'):
    ax.plot(
        c_data['year'],
        c_data['mean_steps'],
        color='#444444',
        alpha=0.35,
        linewidth=0.8,
    )

  group_summary = (
      c_yearly.groupby('year')['mean_steps']
      .agg(['mean', 'std', 'count'])
      .reset_index()
  )
  group_summary['sem'] = group_summary['std'] / np.sqrt(group_summary['count'])
  group_summary['ci95'] = 1.96 * group_summary['sem']

  ax.plot(
      group_summary['year'],
      group_summary['mean'],
      color='#1f77b4',
      linewidth=1.5,
  )
  ax.fill_between(
      group_summary['year'],
      group_summary['mean'] - group_summary['ci95'],
      group_summary['mean'] + group_summary['ci95'],
      color='#1f77b4',
      alpha=0.25,
  )
  ax.set_title(title_str)

age_tuples = [
    ('Young Adults (20-39)', [25, 35], ax_age_1, 'B1. Ages 20–39'),
    ('Middle-Aged (40-59)', [45, 55], ax_age_2, 'B2. Ages 40–59'),
    ('Older Adults (60+)', [65, 75], ax_age_3, 'B3. Ages 60+'),
]

for g_name, midages, ax, title_str in age_tuples:
  sub_df = fb_df[
      (fb_df['country'].isin(target_countries))
      & (fb_df['year'] <= 2025)
      & (fb_df['midage_10'].isin(midages))
  ].copy()
  c_yearly = (
      sub_df.groupby(['country', 'year'])
      .agg(total_steps=('steps', 'sum'), total_readings=('readings', 'sum'))
      .reset_index()
  )
  c_yearly['mean_steps'] = c_yearly['total_steps'] / c_yearly['total_readings']

  for country, c_data in c_yearly.groupby('country'):
    ax.plot(
        c_data['year'],
        c_data['mean_steps'],
        color='#444444',
        alpha=0.35,
        linewidth=0.8,
    )

  group_summary = (
      c_yearly.groupby('year')['mean_steps']
      .agg(['mean', 'std', 'count'])
      .reset_index()
  )
  group_summary['sem'] = group_summary['std'] / np.sqrt(group_summary['count'])
  group_summary['ci95'] = 1.96 * group_summary['sem']

  ax.plot(
      group_summary['year'],
      group_summary['mean'],
      color='#1f77b4',
      linewidth=1.5,
  )
  ax.fill_between(
      group_summary['year'],
      group_summary['mean'] - group_summary['ci95'],
      group_summary['mean'] + group_summary['ci95'],
      color='#1f77b4',
      alpha=0.25,
  )
  ax.set_title(title_str)

for ax in all_demo_axes:
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.set_xticks([2016, 2020, 2024])
  ax.set_xticklabels(['2015', '2020', '2025'])
  ax.set_xlabel('Year')
  if ax in leftmost_demo:
    ax.set_ylabel('Mean\nDaily\nSteps', rotation=0, ha='right', va='center')
  else:
    ax.tick_params(labelleft=False)

fig.suptitle('Ten year step trajectories by sex and age group')
plt.show()

# Geographic Region Step Trajectories
REGION_MAP = {
    'Australia': 'Asia-Pacific',
    'Austria': 'Europe',
    'Belgium': 'Europe',
    'Canada': 'Americas',
    'Chile': 'Americas',
    'Czech Republic': 'Europe',
    'Czechia': 'Europe',
    'Denmark': 'Europe',
    'Finland': 'Europe',
    'France': 'Europe',
    'Germany': 'Europe',
    'India': 'Asia-Pacific',
    'Ireland': 'Europe',
    'Italy': 'Europe',
    'Japan': 'Asia-Pacific',
    'Mexico': 'Americas',
    'Netherlands': 'Europe',
    'New Zealand': 'Asia-Pacific',
    'Norway': 'Europe',
    'Philippines': 'Asia-Pacific',
    'Poland': 'Europe',
    'Portugal': 'Europe',
    'South Africa': 'South Africa',
    'Spain': 'Europe',
    'Sweden': 'Europe',
    'Switzerland': 'Europe',
    'United Kingdom': 'Europe',
    'UK': 'Europe',
    'United States of America': 'Americas',
    'USA': 'Americas',
}
fb_df['region'] = fb_df['country'].map(REGION_MAP).fillna('Other')

fig_reg, axs_reg = plt.subplots(1, 4, figsize=(7.0, 2.6), sharey=True)

reg_tuples = [
    ('Europe', axs_reg[0], 'Europe\n(N=16)'),
    ('Americas', axs_reg[1], 'Americas\n(N=4)'),
    ('Asia-Pacific', axs_reg[2], 'Asia-Pacific\n(N=5)'),
    ('South Africa', axs_reg[3], 'South Africa\n(N=1)'),
]

for region, ax, title_str in reg_tuples:
  sub_df = fb_df[
      (fb_df['country'].isin(target_countries))
      & (fb_df['year'] <= 2025)
      & (fb_df['region'] == region)
  ].copy()
  c_yearly = (
      sub_df.groupby(['country', 'year'])
      .agg(total_steps=('steps', 'sum'), total_readings=('readings', 'sum'))
      .reset_index()
  )
  c_yearly['mean_steps'] = c_yearly['total_steps'] / c_yearly['total_readings']

  for country, c_data in c_yearly.groupby('country'):
    ax.plot(
        c_data['year'],
        c_data['mean_steps'],
        color='#444444',
        alpha=0.35,
        linewidth=0.8,
    )

  if not c_yearly.empty:
    group_summary = (
        c_yearly.groupby('year')['mean_steps']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    group_summary['sem'] = group_summary['std'] / np.sqrt(
        np.maximum(group_summary['count'], 1)
    )
    group_summary['ci95'] = 1.96 * group_summary['sem']

    ax.plot(
        group_summary['year'],
        group_summary['mean'],
        color='#1f77b4',
        linewidth=1.5,
    )
    ax.fill_between(
        group_summary['year'],
        group_summary['mean'] - group_summary['ci95'],
        group_summary['mean'] + group_summary['ci95'],
        color='#1f77b4',
        alpha=0.25,
    )

  ax.set_title(title_str, fontsize=7.5)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.set_xticks([2016, 2020, 2024])
  ax.set_xticklabels(['2015', '2020', '2025'], fontsize=6.5)
  ax.set_xlabel('Year', fontsize=7.5)

  if ax == axs_reg[0]:
    ax.set_ylabel('Mean\nDaily\nSteps', rotation=0, ha='right', va='center')
  else:
    ax.tick_params(labelleft=False)

fig_reg.suptitle('Ten year step trajectories by geographic region')
plt.tight_layout()
plt.show()

INCOME_MAP = {
    'Australia': 'High Income',
    'Austria': 'High Income',
    'Belgium': 'High Income',
    'Canada': 'High Income',
    'Chile': 'High Income',
    'Czech Republic': 'High Income',
    'Czechia': 'High Income',
    'Denmark': 'High Income',
    'Finland': 'High Income',
    'France': 'High Income',
    'Germany': 'High Income',
    'India': 'Middle Income',
    'Ireland': 'High Income',
    'Israel': 'High Income',
    'Italy': 'High Income',
    'Japan': 'High Income',
    'Mexico': 'Middle Income',
    'Netherlands': 'High Income',
    'New Zealand': 'High Income',
    'Norway': 'High Income',
    'Philippines': 'Middle Income',
    'Poland': 'High Income',
    'Portugal': 'High Income',
    'South Africa': 'Middle Income',
    'Spain': 'High Income',
    'Sweden': 'High Income',
    'Switzerland': 'High Income',
    'United Kingdom': 'High Income',
    'UK': 'High Income',
    'United States of America': 'High Income',
    'USA': 'High Income',
}
fb_df['income_group'] = fb_df['country'].map(INCOME_MAP).fillna('Other')

fig_inc, axs_inc = plt.subplots(1, 2, figsize=(7.0, 2.6), sharey=True)

inc_tuples = [
    ('High Income', axs_inc[0], 'High Income\n(N=22)'),
    ('Middle Income', axs_inc[1], 'Middle Income\n(N=6)'),
]

for inc, ax, title_str in inc_tuples:
  sub_df = fb_df[
      (fb_df['country'].isin(target_countries))
      & (fb_df['year'] <= 2025)
      & (fb_df['income_group'] == inc)
  ].copy()
  c_yearly = (
      sub_df.groupby(['country', 'year'])
      .agg(total_steps=('steps', 'sum'), total_readings=('readings', 'sum'))
      .reset_index()
  )
  c_yearly['mean_steps'] = c_yearly['total_steps'] / c_yearly['total_readings']

  for country, c_data in c_yearly.groupby('country'):
    ax.plot(
        c_data['year'],
        c_data['mean_steps'],
        color='#444444',
        alpha=0.35,
        linewidth=0.8,
    )

  group_summary = (
      c_yearly.groupby('year')['mean_steps']
      .agg(['mean', 'std', 'count'])
      .reset_index()
  )
  group_summary['sem'] = group_summary['std'] / np.sqrt(
      np.maximum(group_summary['count'], 1)
  )
  group_summary['ci95'] = 1.96 * group_summary['sem']

  ax.plot(
      group_summary['year'],
      group_summary['mean'],
      color='#1f77b4',
      linewidth=1.5,
  )
  ax.fill_between(
      group_summary['year'],
      group_summary['mean'] - group_summary['ci95'],
      group_summary['mean'] + group_summary['ci95'],
      color='#1f77b4',
      alpha=0.25,
  )

  ax.set_title(title_str, fontsize=8.0)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.set_xticks([2016, 2020, 2024])
  ax.set_xticklabels(['2015', '2020', '2025'])
  ax.set_xlabel('Year')

  if ax == axs_inc[0]:
    ax.set_ylabel('Mean\nDaily\nSteps', rotation=0, ha='right', va='center')
  else:
    ax.tick_params(labelleft=False)

fig_inc.suptitle('Ten year step trajectories by World Bank income group')
plt.tight_layout()
plt.show()

country_trend_rows = []

for country in target_countries:
  c_df = fb_df[(fb_df['country'] == country) & (fb_df['year'] <= 2025)].copy()
  if c_df.empty:
    continue

  c_yearly = (
      c_df.groupby('year')
      .agg(total_steps=('steps', 'sum'), total_readings=('readings', 'sum'))
      .reset_index()
  )
  c_yearly['mean_steps'] = c_yearly['total_steps'] / c_yearly['total_readings']
  c_yearly = c_yearly.sort_values('year').set_index('year')['mean_steps']

  val_2015 = c_yearly.loc[2015] if 2015 in c_yearly.index else np.nan
  val_2020 = c_yearly.loc[2020] if 2020 in c_yearly.index else np.nan
  val_2025 = c_yearly.loc[2025] if 2025 in c_yearly.index else np.nan
  avg_10yr = c_yearly.mean()

  valid_yrs = c_yearly.dropna().index.values
  valid_vals = c_yearly.dropna().values
  if len(valid_yrs) >= 3:
    slope, intercept, r_val, p_val, std_err = stats.linregress(
        valid_yrs, valid_vals
    )
    trend_slope = slope
  else:
    trend_slope = np.nan

  if not np.isnan(val_2015) and not np.isnan(val_2025):
    abs_change_10yr = val_2025 - val_2015
    rel_change_10yr = (val_2025 - val_2015) / val_2015 * 100.0
  elif len(valid_vals) >= 2:
    first_yr = valid_yrs[0]
    last_yr = valid_yrs[-1]
    abs_change_10yr = c_yearly.loc[last_yr] - c_yearly.loc[first_yr]
    rel_change_10yr = (
        (c_yearly.loc[last_yr] - c_yearly.loc[first_yr])
        / c_yearly.loc[first_yr]
        * 100.0
    )
  else:
    abs_change_10yr = np.nan
    rel_change_10yr = np.nan

  region = REGION_MAP.get(country, 'Other')
  income = INCOME_MAP.get(country, 'Other')

  country_trend_rows.append({
      'Country': country,
      'Region': region,
      'Income Group': income,
      '2015 Mean Steps': (
          round(val_2015, 0) if not np.isnan(val_2015) else np.nan
      ),
      '2020 Mean Steps': (
          round(val_2020, 0) if not np.isnan(val_2020) else np.nan
      ),
      '2025 Mean Steps': (
          round(val_2025, 0) if not np.isnan(val_2025) else np.nan
      ),
      '10-Yr Change (Δ steps)': (
          round(abs_change_10yr, 0) if not np.isnan(abs_change_10yr) else np.nan
      ),
      '10-Yr Relative Change (%)': (
          f'{rel_change_10yr:+.1f}%' if not np.isnan(rel_change_10yr) else 'N/A'
      ),
      'Annual Trend (steps/yr)': (
          f'{trend_slope:+.1f}' if not np.isnan(trend_slope) else 'N/A'
      ),
      '10-Yr Overall Mean': round(avg_10yr, 0),
  })

df_country_trends_2015_2025 = pd.DataFrame(country_trend_rows)
df_country_trends_2015_2025 = df_country_trends_2015_2025.sort_values(
    '10-Yr Change (Δ steps)', ascending=False
).reset_index(drop=True)

print('\n' + '=' * 85)
print(
    '=== COUNTRY-BY-COUNTRY 10-YEAR STEP TRAJECTORIES & TREND TABLE (2015–2025)'
    ' ==='
)
print('=' * 85)
display(df_country_trends_2015_2025)

# *FIN*